# Notebook 03: VOCALS-REx Validation and Figure Generation

**Paper 3** — Physics-Informed Neural Networks for Cloud Droplet Profile Retrieval  
Andrew J. Buggee, LASP / CU Boulder

This notebook:
1. Loads the trained retrieval network
2. Applies it to the three MODIS scenes from Papers 1 & 2 (Fig. 3a-c)
3. Compares retrieved profiles with VOCALS-REx in situ measurements
4. Compares with Gauss-Newton retrievals from Papers 1 & 2
5. Generates publication-quality figures

The three cases from Papers 1 & 2 (Table 2):
- Fig 3a: Nov 11, 2008, τ_in-situ ≈ 6.5
- Fig 3b: Nov 11, 2008, τ_in-situ ≈ 11
- Fig 3c: Nov 9, 2008,  τ_in-situ ≈ 19.5

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import torch
from pathlib import Path
import sys

sys.path.insert(0, str(Path('.').resolve().parent))
from src.models import DropletProfileNetwork, RetrievalConfig
from src.data import adiabatic_profile, MODIS_WAVELENGTHS

# Publication figure settings
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
})

device = torch.device('cpu')  # Use CPU for inference/plotting

## 1. Load Trained Model

In [ ]:
# Load the best checkpoint
CHECKPOINT_PATH = '../checkpoints/best_model_notebook.pt'
# Or from Alpine: '../checkpoints/YYYYMMDD_HHMMSS/best_model.pt'

config = RetrievalConfig(
    n_wavelengths=7,
    n_geometry_inputs=4,
    n_levels=10,
    hidden_dims=(256, 256, 256, 256),
    dropout=0.1,
    activation='gelu',
)

model = DropletProfileNetwork(config).to(device)
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device, weights_only=True))
model.eval()
print("Model loaded successfully.")

## 2. Load MODIS and VOCALS-REx Data

Load the same MODIS pixels and in situ profiles used in Papers 1 & 2.

**TODO:** Adapt the paths below to point to your MODIS L1B reflectances
and VOCALS-REx CDP data. You can use your existing MATLAB code to
extract these and save as .npz or .mat files.

In [ ]:
# ===================================================================
# TODO: Load real MODIS reflectances and VOCALS-REx in situ data
# ===================================================================
# 
# For now, we use placeholder data matching the cases from Paper 1.
# Replace these with actual data from your MATLAB pipeline.
#
# Real data loading would look like:
#   import scipy.io as sio
#   case_a = sio.loadmat('path/to/vocals_case_a.mat')
#   modis_refl_a = case_a['reflectances']  # shape (7,)
#   insitu_re_a = case_a['r_eff']           # shape (n_points,)
#   insitu_alt_a = case_a['altitude']       # shape (n_points,)
# ===================================================================

# Placeholder cases based on Paper 1 Fig. 3 values
cases = {
    '3a': {
        'label': 'Nov 11, 2008 (τ ≈ 6.5)',
        'tau_insitu': 6.5,
        'lwp_insitu': 21.5,
        'lwp_modis': 34.0,
        'modis_re_2.1': 9.07,
        'modis_tau_2.1': 5.98,
        # Approximate in situ profile from Fig 3a
        'insitu_re': np.array([9.5, 8.5, 7.8, 7.2, 6.9, 6.5, 6.2, 5.8, 5.3, 4.5]),
        'insitu_re_err': np.array([0.95, 0.85, 0.78, 0.72, 0.69, 0.65, 0.62, 0.58, 0.53, 0.9]),
        'sza': 27.0,
        'vza': 14.0,
        'phi': 45.0,
        'wind_speed': 6.0,
    },
    '3b': {
        'label': 'Nov 11, 2008 (τ ≈ 11)',
        'tau_insitu': 11.0,
        'lwp_insitu': 37.2,
        'lwp_modis': 50.0,
        'modis_re_2.1': 8.21,
        'modis_tau_2.1': 9.69,
        'insitu_re': np.array([7.5, 7.0, 6.5, 6.0, 5.6, 5.2, 4.8, 4.3, 3.8, 3.2]),
        'insitu_re_err': np.array([0.75, 0.70, 0.65, 0.60, 0.56, 0.52, 0.48, 0.43, 0.76, 0.64]),
        'sza': 22.0,
        'vza': 8.0,
        'phi': 120.0,
        'wind_speed': 7.0,
    },
    '3c': {
        'label': 'Nov 9, 2008 (τ ≈ 19.5)',
        'tau_insitu': 19.5,
        'lwp_insitu': 61.9,
        'lwp_modis': 95.0,
        'modis_re_2.1': 7.71,
        'modis_tau_2.1': 19.51,
        'insitu_re': np.array([12.5, 11.5, 10.5, 9.8, 9.0, 8.2, 7.5, 6.8, 6.0, 5.0]),
        'insitu_re_err': np.array([1.25, 1.15, 1.05, 0.98, 0.90, 0.82, 0.75, 0.68, 0.60, 1.0]),
        'sza': 24.0,
        'vza': 10.0,
        'phi': 90.0,
        'wind_speed': 5.5,
    },
}

print(f"Loaded {len(cases)} validation cases")

## 3. Run PINN Retrieval

**TODO:** When you have real MODIS reflectances, normalize them using
the same statistics from the training dataset. For now we skip this
step since we don't have real reflectance inputs.

In [ ]:
def retrieve_profile(model, reflectances, sza, vza, phi, wind_speed,
                     dataset=None, device='cpu'):
    """
    Run the retrieval network on a single pixel.
    
    Parameters
    ----------
    model : DropletProfileNetwork (eval mode)
    reflectances : np.ndarray, shape (n_wavelengths,) — TOA reflectances
    sza, vza, phi, wind_speed : float — observation geometry
    dataset : LibRadtranDataset — for normalization statistics
    
    Returns
    -------
    profile : np.ndarray, shape (n_levels,) — r_e in μm
    tau_c : float — cloud optical depth
    """
    # TODO: Implement proper normalization using dataset statistics
    # For now, this is a placeholder
    
    # Normalize inputs (you'd use dataset._normalize_input in practice)
    x = np.concatenate([reflectances, [sza/80, vza/60, phi/180, wind_speed/15]])
    x_tensor = torch.tensor(x, dtype=torch.float32).unsqueeze(0).to(device)
    
    with torch.no_grad():
        output = model(x_tensor)
    
    profile = output['profile'].squeeze(0).cpu().numpy()
    tau_c = output['tau_c'].squeeze().cpu().item()
    
    return profile, tau_c

print("Retrieval function defined. Ready for real MODIS data.")

## 4. Publication Figure: PINN vs In Situ vs Gauss-Newton

This figure will be the key comparison figure for Paper 3,
analogous to Fig. 3 from Paper 1. It shows all three methods
against in situ data.

In [ ]:
# ===================================================================
# Publication figure template
# ===================================================================
# When you have real retrieval results, populate pinn_re and pinn_tau
# for each case. For now, this shows the figure layout with in situ
# data and MODIS bispectral values from Paper 1.
# ===================================================================

fig, axes = plt.subplots(1, 3, figsize=(14, 5.5), sharey=True)
z_norm = np.linspace(0, 1, 10)  # 0 = cloud top, 1 = cloud base

for ax, (case_id, case) in zip(axes, cases.items()):
    # In situ measurements
    ax.errorbar(case['insitu_re'], z_norm, xerr=case['insitu_re_err'],
                fmt='ko', markersize=4, linewidth=1, capsize=2,
                label='In situ (CDP)', zorder=3)
    
    # MODIS bispectral retrieval (vertical line)
    ax.axvline(case['modis_re_2.1'], color='#3B82F6', linestyle=':',
               linewidth=1.5, alpha=0.8, label=f"MODIS r$_{{2.1}}$ = {case['modis_re_2.1']} μm")
    
    # Adiabatic reference (using in situ top & base)
    r_adiab = adiabatic_profile(case['insitu_re'][0], case['insitu_re'][-1], 10)
    ax.plot(r_adiab, z_norm, '--', color='gray', linewidth=1, alpha=0.5, label='Adiabatic ref.')
    
    # PINN retrieval (placeholder — fill in with real results)
    # pinn_re, pinn_tau = retrieve_profile(model, modis_refl, ...)
    # ax.plot(pinn_re, z_norm, 's-', color='#10B981', markersize=4,
    #         linewidth=1.5, label='PINN retrieval')
    
    # Gauss-Newton retrieval from Paper 1 (placeholder — load from your data)
    # ax.plot(gn_profile, z_norm_gn, '-.', color='#F43F5E', linewidth=1.5,
    #         label='Gauss-Newton (Paper 1)')
    
    ax.set_xlabel('Effective Radius (μm)', fontsize=11)
    ax.set_title(f'({case_id}) {case["label"]}', fontsize=11)
    ax.legend(fontsize=8, loc='lower right')
    ax.grid(True, alpha=0.15)
    
    # Add LWP annotation box
    text = (f"LWP$_{{in\\,situ}}$ = {case['lwp_insitu']} g/m²\n"
            f"LWP$_{{MODIS}}$ = {case['lwp_modis']} g/m²")
    ax.text(0.03, 0.03, text, transform=ax.transAxes, fontsize=8,
            verticalalignment='bottom', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

axes[0].set_ylabel('Normalized Depth (0=top, 1=base)', fontsize=11)
axes[0].invert_yaxis()

plt.suptitle('Droplet Profile Retrieval Comparison', fontsize=14, y=1.02)
plt.tight_layout()
# plt.savefig('../figures/fig_vocals_comparison.pdf')
plt.show()

## 5. LWP Comparison Table

Compare liquid water path estimates: in situ, MODIS bispectral,
Gauss-Newton (Paper 1), and PINN (this work).

In [ ]:
def compute_lwp(profile_re, tau_c, n_levels, cloud_thickness_m=500):
    """
    Estimate LWP from a retrieved droplet profile.
    
    LWP = integral of LWC dz
    LWC = (4/3) * pi * rho_w * N_c * r_e^3  (Eq. 2 from Paper 1)
    
    For a simplified estimate assuming constant N_c:
    LWP proportional to tau_c * r_e_effective / (3 * Q_ext)
    
    Using the standard formula: LWP = (2/3) * rho_w * tau_c * r_e_eff
    where r_e_eff is the column-weighted effective radius.
    """
    # Simple trapezoidal integration of r_e profile
    dz = cloud_thickness_m / (n_levels - 1)  # meters
    
    # Weight-averaged r_e (in meters)
    r_e_mean = np.trapz(profile_re, dx=1/(n_levels-1)) * 1e-6  # μm to m
    
    # LWP = (2/3) * rho_w * tau * r_eff
    rho_w = 1e6  # g/m³
    lwp = (2/3) * rho_w * tau_c * r_e_mean  # g/m²
    
    return lwp

print("LWP computation ready.")
print("TODO: Compute LWP for PINN retrievals once real data is available.")

## 6. Solution Uniqueness Analysis

One of the key questions from Papers 1 & 2: how unique is the retrieved
solution? For the PINN, we can assess this by:
1. Adding noise to inputs and measuring output variance (ensemble approach)
2. Using Monte Carlo dropout for approximate uncertainty

In [ ]:
def ensemble_uncertainty(model, x_input, n_perturbations=100,
                         noise_level=0.02, device='cpu'):
    """
    Estimate retrieval uncertainty by perturbing input reflectances.
    
    This is analogous to the solution space analysis in Figs. 8-9
    of Paper 1, but computed via Monte Carlo sampling rather than
    exhaustive grid search.
    
    Parameters
    ----------
    model : trained DropletProfileNetwork
    x_input : torch.Tensor, shape (1, input_dim) — normalized input
    n_perturbations : int — number of noisy copies
    noise_level : float — relative noise standard deviation
    
    Returns
    -------
    profile_mean : (n_levels,) — mean predicted profile
    profile_std : (n_levels,) — standard deviation at each level
    tau_mean, tau_std : float — mean and std of predicted tau
    """
    model.eval()
    profiles = []
    taus = []
    
    with torch.no_grad():
        for _ in range(n_perturbations):
            noise = torch.randn_like(x_input) * noise_level
            x_noisy = x_input + noise
            output = model(x_noisy.to(device))
            profiles.append(output['profile'].cpu().numpy())
            taus.append(output['tau_c'].cpu().item())
    
    profiles = np.array(profiles).squeeze()  # (n_perturbations, n_levels)
    taus = np.array(taus)
    
    return (
        profiles.mean(axis=0), profiles.std(axis=0),
        taus.mean(), taus.std(),
    )

print("Uncertainty estimation function defined.")
print("Run on real MODIS data to assess solution uniqueness.")

In [ ]:
# Placeholder: figure showing uncertainty as function of cloud depth
# This will be an important result — we expect larger uncertainty
# at cloud base, consistent with the weighting functions from Paper 1

# fig, ax = plt.subplots(1, 1, figsize=(6, 5))
# ax.fill_betweenx(z_norm, profile_mean - 2*profile_std,
#                   profile_mean + 2*profile_std, alpha=0.2, color='#10B981')
# ax.plot(profile_mean, z_norm, '-', color='#10B981', linewidth=2, label='PINN mean')
# ax.errorbar(insitu_re, z_norm, xerr=insitu_err, fmt='ko', label='In situ')
# ax.set_xlabel('Effective Radius (μm)')
# ax.set_ylabel('Normalized Depth')
# ax.invert_yaxis()
# ax.legend()
# plt.show()